<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/wk-10.B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install necessary libraries
!pip install -q transformers[torch] datasets

import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)

# 2. Define a "Creative Writing" dataset
# This set of prompts/stories teaches the model a specific "poetic" tone.
creative_data = [
    "Whispers of the ancient forest danced among the emerald leaves, telling secrets of old.",
    "The silver moonlight bathed the ocean in a ghostly glow, turning waves into liquid diamonds.",
    "Time stood still in the quiet village, where the scent of jasmine hung heavy in the midnight air.",
    "A symphony of colors painted the twilight sky as the sun dipped below the jagged horizon.",
    "Beneath the weeping willow, the shadows stretched like long fingers reaching for the forgotten path."
]

# Convert to Dataset object
dataset = Dataset.from_dict({"text": creative_data})

# 3. Load GPT-2 Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Essential for GPT-2
model = GPT2LMHeadModel.from_pretrained(model_name)

# 4. Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./creative-gpt",
    num_train_epochs=25,          # High epochs for a tiny dataset to ensure influence
    per_device_train_batch_size=2,
    save_strategy="no",           # Don't save files to disk to keep Colab clean
    report_to="none",
    learning_rate=5e-5            # Fine-tuning learning rate
)

# 6. Fine-Tuning Process
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("--- Starting Fine-Tuning for Creative Tone ---")
trainer.train()
print("Fine-tuning complete!")

# 7. Generate and Display Creative Output
print("\n--- RESULTS: GENERATED CREATIVE TEXT ---")
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Try a prompt that would usually be answered plainly
prompt = "The old clock"
result = generator(prompt, max_length=60, do_sample=True, temperature=0.8, truncation=True)

print(f"Prompt: {prompt}")
print(f"Generated Text: {result[0]['generated_text']}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

--- Starting Fine-Tuning for Creative Tone ---


Step,Training Loss


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuning complete!

--- RESULTS: GENERATED CREATIVE TEXT ---
Prompt: The old clock
Generated Text: The old clock struck the twilight sky, revealing the ancient jagged horizon in the distance. The jagged horizon stretched like a long, long, curved line.

The jagged horizon stretched like a long, curved line.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.
The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched like long fingers reaching for the forgotten path.

The shadows stretched 